# Setup

In [1]:
import pandas as pd
import glob
import os
import numpy as np

In [2]:
def convert_price_columns(df, price_cols=['open', 'high', 'low', 'close']):
    for col in price_cols:
        df[col] = df[col].astype(float) * 1000
        df[col] = df[col].astype(int)
    return df

In [3]:
files = glob.glob("new/*.csv")

list_df = []
for file in files:
    df_temp = pd.read_csv(file)
    df_temp = convert_price_columns(df_temp)
    symbol = os.path.basename(file).split('.')[0]
    df_temp['Symbol'] = symbol
    list_df.append(df_temp)

df = pd.concat(list_df, ignore_index=True)

In [11]:
df.head()

,time,open,high,low,close,volume,Symbol
0,2018-01-02,17560,18330,17530,18330,5022160,FPT_quote_history
1,2018-01-03,18540,18600,18240,18330,2829930,FPT_quote_history
2,2018-01-04,18390,18700,18360,18700,2784800,FPT_quote_history
3,2018-01-05,18700,18730,18360,18390,2851450,FPT_quote_history
4,2018-01-08,18270,18670,18180,18670,2529030,FPT_quote_history


In [6]:
df.isna().sum()

time      0
open      0
high      0
low       0
close     0
volume    0
Symbol    0
dtype: int64

In [13]:
len(df)

6000

In [10]:
df['time'] = pd.to_datetime(df['time'])

Fill data with linear interpolation

In [20]:
full_range = pd.date_range(start='2018-01-01', end='2025-01-31', freq='D')
numeric_cols = ['open', 'high', 'low', 'close', 'volume']
filled_list = []

df['time'] = pd.to_datetime(df['time'], format='%Y-%m-%d')

for symbol, group in df.groupby('Symbol'):
    group = group.sort_values('time').drop_duplicates(subset='time', keep='first')
    group = group.set_index('time')
    
    group = group.reindex(full_range)
    group.index.name = 'time'
    group['Symbol'] = symbol  
    
    group[numeric_cols] = group[numeric_cols].interpolate(method='linear').ffill().bfill()
    
    group = group.reset_index()
    filled_list.append(group)

df = pd.concat(filled_list, ignore_index=True)

In [25]:
len(df)    

7764

In [27]:
df.head(20)

,time,open,high,low,close,volume,Symbol,OBV,RSI6,RSI12
0,2018-01-01,17560.000000,18330.000000,17530.000000,18330.000000,5.022160e+06,FPT_quote_history,0.000000e+00,NaN,NaN
1,2018-01-02,17560.000000,18330.000000,17530.000000,18330.000000,5.022160e+06,FPT_quote_history,0.000000e+00,NaN,NaN
2,2018-01-03,18540.000000,18600.000000,18240.000000,18330.000000,2.829930e+06,FPT_quote_history,0.000000e+00,NaN,NaN
3,2018-01-04,18390.000000,18700.000000,18360.000000,18700.000000,2.784800e+06,FPT_quote_history,2.784800e+06,NaN,NaN
4,2018-01-05,18700.000000,18730.000000,18360.000000,18390.000000,2.851450e+06,FPT_quote_history,-6.665000e+04,NaN,NaN
5,2018-01-06,18556.666667,18710.000000,18300.000000,18483.333333,2.743977e+06,FPT_quote_history,2.677327e+06,NaN,NaN
6,2018-01-07,18413.333333,18690.000000,18240.000000,18576.666667,2.636503e+06,FPT_quote_history,5.313830e+06,64.230769,NaN
7,2018-01-08,18270.000000,18670.000000,18180.000000,18670.000000,2.529030e+06,FPT_quote_history,7.842860e+06,67.708333,NaN
8,2018-01-09,18700.000000,19100.000000,18480.000000,19030.000000,4.010010e+06,FPT_quote_history,1.185287e+07,76.515152,NaN
9,2018-01-10,19000.000000,19310.000000,18730.000000,18850.000000,2.610670e+06,FPT_quote_history,9.242200e+06,56.637168,NaN


In [23]:
df['OBV'] = np.where(df['close'] > df['close'].shift(1), df['volume'], np.where(df['close'] < df['close'].shift(1), -df['volume'], 0))
df['OBV'] = df['OBV'].cumsum()  

In [24]:
def RSI(series, period):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(window=period, min_periods=period).mean()
    avg_loss = loss.rolling(window=period, min_periods=period).mean()
    rsi = 100 * avg_gain / (avg_gain + avg_loss)
    return rsi

df['RSI6'] = RSI(df['close'], 6)
df['RSI12'] = RSI(df['close'], 12)

In [28]:
df['SMA3'] = df['close'].rolling(window=3, min_periods=1).mean()

In [29]:
df['EMA6'] = df['close'].ewm(span=6, adjust=False).mean()
df['EMA12'] = df['close'].ewm(span=12, adjust=False).mean()

In [30]:
def compute_ATR_wilder(df, period=14):
    high = df['high']
    low = df['low']
    close = df['close']
    
    prev_close = close.shift(1)
    tr1 = high - low
    tr2 = (high - prev_close).abs()
    tr3 = (low - prev_close).abs()
    
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.ewm(alpha=1/period, adjust=False).mean()
    
    return atr

df['ATR14'] = compute_ATR_wilder(df, period=14)

In [32]:
def MFI(df, period=14):
    typical_price = (df['high'] + df['low'] + df['close'] / 3)
    mf = typical_price * df['volume']
    delta_tp = typical_price.diff()
    positive_mf = mf.where(delta_tp > 0, 0)
    negative_mf = mf.where(delta_tp < 0, 0).abs()
    pos_mf_sum = positive_mf.rolling(window=period, min_periods=period).sum()
    neg_mf_sum = negative_mf.rolling(window=period, min_periods=period).sum()
    mfi = 100 - (100 / (1 + pos_mf_sum / neg_mf_sum))
    return mfi

df['MFI14'] = MFI(df, period=14)

In [34]:
def ADX(df, period):
    up_move = df['high'] - df['high'].shift(1)
    down_move = df['low'].shift(1) - df['low']
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0)
    tr = pd.concat([
        (df['high'] - df['low']),
        (df['high'] - df['close'].shift(1)).abs(),
        (df['low'] - df['close'].shift(1)).abs()
    ], axis=1).max(axis=1)
    atr = tr.rolling(window=period, min_periods=period).mean()
    plus_di = 100 * (pd.Series(plus_dm).rolling(window=period, min_periods=period).sum() / atr)
    minus_di = 100 * (pd.Series(minus_dm).rolling(window=period, min_periods=period).sum() / atr)
    dx = 100 * (abs(plus_di - minus_di) / (plus_di + minus_di))
    adx = dx.rolling(window=period, min_periods=period).mean()
    return adx
df['ADX14'] = ADX(df, period=14)
df['ADX20'] = ADX(df, period=20)

In [36]:
df['MOM1'] = df['close'] - df['close'].shift(1)
df['MOM3'] = df['close'] - df['close'].shift(3)

In [37]:
def CCI(df, period):
    tp = (df['high'] + df['low'] + df['close']) / 3
    sma_tp = tp.rolling(window=period, min_periods=period).mean()
    md = tp.rolling(window=period, min_periods=period).apply(lambda x: np.mean(np.abs(x - np.mean(x))), raw=True)
    cci = (tp - sma_tp) / (0.15 * md)
    return cci
df['CCI12'] = CCI(df, period=12)
df['CCI20'] = CCI(df, period=20)

In [38]:
df['ROCR3'] = (df['close'] / df['close'].shift(3)) * 100
df['ROCR12'] = (df['close'] / df['close'].shift(12)) * 100

In [39]:
ema12 = df['close'].ewm(span=12, adjust=False).mean()
ema26 = df['close'].ewm(span=26, adjust=False).mean()
df['outMACD'] = ema12 - ema26
df['outMACDSignal'] = df['outMACD'].ewm(span=9, adjust=False).mean()
df['outMACDHist'] = df['outMACD'] - df['outMACDSignal']

In [40]:
highest_high = df['high'].rolling(window=10, min_periods=10).max()
lowest_low = df['low'].rolling(window=10, min_periods=10).min()
df['WILLR'] = ((highest_high - df['close']) / (highest_high - lowest_low)) * 100

In [49]:
df.isna().sum()[df.isna().sum() > 0]

Series([], dtype: int64)

In [42]:
def TSF(series, period):
    def linreg(x):
        idx = np.arange(len(x))
        slope, intercept = np.polyfit(idx, x, 1)
        return slope * len(x) + intercept
    return series.rolling(window=period, min_periods=period).apply(linreg, raw=True)

df['TSF10'] = TSF(df['close'], 10)
df['TSF20'] = TSF(df['close'], 20)

In [44]:
ema1 = df['close'].ewm(span=15, adjust=False).mean()
ema2 = ema1.ewm(span=15, adjust=False).mean()
ema3 = ema2.ewm(span=15, adjust=False).mean()
df['TRIX'] = (ema3 - ema3.shift(1)) / ema3.shift(1) * 100

In [46]:
df['BBANDSMIDDLE'] = df['close'].rolling(window=21, min_periods=21).mean()
rolling_std = df['close'].rolling(window=21, min_periods=21).std()
df['BBANDSUPPER'] = df['BBANDSMIDDLE'] + 2 * rolling_std
df['BBANDSLOWER'] = df['BBANDSMIDDLE'] - 2 * rolling_std

In [48]:
# Fill missing values with forward and backward fill
df = df.ffill().bfill()

In [52]:
df.columns

Index(['time', 'open', 'high', 'low', 'close', 'volume', 'Symbol', 'OBV',
       'RSI6', 'RSI12', 'SMA3', 'EMA6', 'EMA12', 'ATR14', 'MFI14', 'ADX14',
       'ADX20', 'MOM1', 'MOM3', 'CCI12', 'CCI20', 'ROCR3', 'ROCR12', 'outMACD',
       'outMACDSignal', 'outMACDHist', 'WILLR', 'TSF10', 'TSF20', 'TRIX',
       'BBANDSMIDDLE', 'BBANDSUPPER', 'BBANDSLOWER'],
      dtype='object')

In [54]:
len(df.columns)

33

In [55]:
len(df)

7764

In [57]:
# Đổi tên các giá trị trong cột Symbol
df['Symbol'] = df['Symbol'].replace({
    'FPT_quote_history': 'FPT',
    'VN30_quote_history': 'VN30',
    'VNINDEX_quote_history': 'VNINDEX'
})

In [58]:
df.to_csv('processed_fpt.csv', index=False)

# Archived

In [ ]:
# Define the helper functions used below
def stochastic_oscillator(high, low, close, period=14):
    lowest_low = low.rolling(window=period, min_periods=1).min()
    highest_high = high.rolling(window=period, min_periods=1).max()
    return (close - lowest_low) / (highest_high - lowest_low) * 100

def true_range(high, low, close):
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low - prev_close).abs()
    ], axis=1).max(axis=1)
    return tr

def RSI_ex(series, period=14):
    delta = series.diff(1)
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(span=period, adjust=False).mean()
    avg_loss = loss.ewm(span=period, adjust=False).mean()
    RS = avg_gain / avg_loss
    return 100 - (100 / (1 + RS))

def stochastic_RSI_ex(rsi_series, period=14):
    min_rsi = rsi_series.rolling(window=period, min_periods=1).min()
    max_rsi = rsi_series.rolling(window=period, min_periods=1).max()
    return (rsi_series - min_rsi) / (max_rsi - min_rsi)

def weighted_moving_average(prices, window):
    return prices.rolling(window, min_periods=1).apply(
        lambda x: np.dot(x, np.arange(1, len(x)+1)) / np.sum(np.arange(1, len(x)+1)),
        raw=True
    )
    
def calculate_adx(high, low, close, period=14):
    prev_high = high.shift(1)
    prev_low = low.shift(1)
    prev_close = close.shift(1)
    up_move = high - prev_high
    down_move = prev_low - low
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
    tr = true_range(high, low, close)
    atr = tr.rolling(window=period, min_periods=period).sum()
    plus_dm_sum = pd.Series(plus_dm).rolling(window=period, min_periods=period).sum()
    minus_dm_sum = pd.Series(minus_dm).rolling(window=period, min_periods=period).sum()
    plus_di = 100 * (plus_dm_sum / atr)
    minus_di = 100 * (minus_dm_sum / atr)
    dx = 100 * (np.abs(plus_di - minus_di) / (plus_di + minus_di))
    adx = dx.rolling(window=period, min_periods=period).mean()
    # Here, DMI is defined as the difference between +DI and -DI.
    dmi = plus_di - minus_di
    return adx, plus_di, minus_di, dmi

def MFI(high, low, close, volume, period=14):
    typical_price = (high + low + close) / 3.0
    money_flow = typical_price * volume
    tp_diff = typical_price.diff(1)
    pos_mf = money_flow.where(tp_diff > 0, 0)
    neg_mf = money_flow.where(tp_diff < 0, 0)
    pos_mf_sum = pos_mf.rolling(window=period, min_periods=1).sum()
    neg_mf_sum = neg_mf.rolling(window=period, min_periods=1).sum().abs()
    return 100 * pos_mf_sum / (pos_mf_sum + neg_mf_sum)

def CCI(high, low, close, period):
    tp = (high + low + close) / 3.0
    sma_tp = tp.rolling(window=period, min_periods=1).mean()
    mad = tp.rolling(window=period, min_periods=1).apply(lambda x: np.mean(np.abs(x - np.mean(x))), raw=True)
    return (tp - sma_tp) / (0.015 * mad)

def TRIX(prices, period=15):
    ema1 = prices.ewm(span=period, adjust=False).mean()
    ema2 = ema1.ewm(span=period, adjust=False).mean()
    ema3 = ema2.ewm(span=period, adjust=False).mean()
    return ema3.pct_change() * 100

def WILLR(high, low, close, period=14):
    highest_high = high.rolling(window=period, min_periods=1).max()
    lowest_low = low.rolling(window=period, min_periods=1).min()
    return -100 * (highest_high - close) / (highest_high - lowest_low)

In [ ]:
full_range = pd.date_range(start='2018-01-01', end='2024-12-31', freq='D')

features_list = []

for symbol, group in df.groupby('Symbol'):
    group = group.sort_values('Date').set_index('Date')
    group = group.reindex(full_range)
    group.index.name = 'Date'
    group['Symbol'] = symbol
    
    # Returns based on Close
    group['return_day'] = group['Close'].pct_change()
    group['return_week'] = group['Close'].pct_change(periods=5)
    group['return_month'] = group['Close'].pct_change(periods=22)
    
    # Volatility: rolling std of daily returns
    group['volatility_day'] = group['return_day'].rolling(window=5, min_periods=1).std()
    group['volatility_week'] = group['return_day'].rolling(window=21, min_periods=1).std()
    group['volatility_month'] = group['return_day'].rolling(window=63, min_periods=1).std()
    
    # Liquidity: rolling mean of Volume
    group['liquidity_day'] = group['Volume'].rolling(window=5, min_periods=1).mean()
    group['liquidity_week'] = group['Volume'].rolling(window=21, min_periods=1).mean()
    group['liquidity_month'] = group['Volume'].rolling(window=63, min_periods=1).mean()
    
    # Pressure features
    group['high_minus_close'] = (group['High'] - group['Close']) / group['Open']
    group['low_minus_open'] = (group['Low'] - group['Open']) / group['Open']
    
    # Cumulative return
    group['cumulative_return'] = group['Close'] / group['Close'].iloc[0] - 1
    
    # Stochastic oscillator over 14-day window
    group['Stochastic_Osc'] = stochastic_oscillator(group['High'], group['Low'], group['Close'], period=14)
    
    # ATR (Average True Range) over 14-day window
    group['True_Range'] = true_range(group['High'], group['Low'], group['Close'])
    group['ATR'] = group['True_Range'].rolling(window=14, min_periods=1).mean()
    group.drop('True_Range', axis=1, inplace=True)
    
    # 'ADX14' and 'ADX20'
    # adx14, plus_di_14, minus_di_14, dmi14 = calculate_adx(group['High'], group['Low'], group['Close'], period=14)
    # adx20, plus_di_20, minus_di_20, dmi20 = calculate_adx(group['High'], group['Low'], group['Close'], period=20)
    # group['ADX14'] = adx14
    # group['ADX20'] = adx20
    
    # Simple Moving Averages (SMA) for windows 7, 14, 21, 50, 100
    sma_windows = [3, 7, 14, 21, 50, 100]
    for window in sma_windows:
        group[f'SMA_{window}'] = group['Close'].rolling(window, min_periods=1).mean()
    
    # Weighted Moving Averages (WMA) for windows 7, 14, 21, 50, 100
    wma_windows = [3, 7, 14, 21, 50, 100]
    for window in wma_windows:
        group[f'WMA_{window}'] = weighted_moving_average(group['Close'], window)
    
    # 11. EMA, MACD, and Signal computed using EMAs on Close
    group['EMA6'] = group['Close'].ewm(span=6, adjust=False).mean()
    group['EMA12'] = group['Close'].ewm(span=12, adjust=False).mean()
    group['EMA26'] = group['Close'].ewm(span=26, adjust=False).mean()
    group['outMACD'] = group['EMA12'] - group['EMA26']
    group['outMACDSignal'] = group['outMACD'].ewm(span=9, adjust=False).mean()
    group['outMACDHist'] = group['outMACD'] - group['outMACDSignal']    
    
    # 12. RSI computed on Close (exponential version, 14-day)
    group['RSI6'] = RSI_ex(group['Close'], period=6)
    group['RSI12'] = RSI_ex(group['Close'], period=12)
    group['RSI14'] = RSI_ex(group['Close'], period=14)
    
    # 13. Stochastic RSI on the RSI computed above
    group['StochRSI_6'] = stochastic_RSI_ex(group['RSI6'], 6)
    group['StochRSI_12'] = stochastic_RSI_ex(group['RSI12'], 12)
    group['StochRSI_14'] = stochastic_RSI_ex(group['RSI14'], 14)
    
    # 14. Bollinger Bands (21-day SMA ± 2 std deviations) on Close
    group['BBANDSMIDDLE'] = group['Close'].rolling(window=21, min_periods=1).mean()
    rolling_std = group['Close'].rolling(window=21, min_periods=1).std()
    group['BBANDSUPPER'] = group['BBANDSMIDDLE'] + 2 * rolling_std
    group['BBANDSLOWER'] = group['BBANDSMIDDLE'] - 2 * rolling_std
    
    # on balance volume
    obv = group['Volume'].copy() * 0  # initialize to 0
    price_diff = group['Close'].diff()
    obv = np.where(price_diff > 0, group['Volume'], np.where(price_diff < 0, -group['Volume'], 0))
    group['OBV'] = pd.Series(obv, index=group.index).cumsum()
    
    # money flow index
    group['MFI14'] = MFI(group['High'], group['Low'], group['Close'], group['Volume'], period=14)
    
    # momentum
    group['MOM1'] = group['Close'] - group['Close'].shift(1)
    group['MOM3'] = group['Close'] - group['Close'].shift(3)
    group['MOM7'] = group['Close'] - group['Close'].shift(7)
    
    # 'CCI12' and 'CCI20'
    group['CCI12'] = CCI(group['High'], group['Low'], group['Close'], period=12)
    group['CCI20'] = CCI(group['High'], group['Low'], group['Close'], period=20)
    
    # 'ROCR3' and 'ROCR12' – Rate Of Change Ratio
    group['ROCR3'] = group['Close'] / group['Close'].shift(3)
    group['ROCR12'] = group['Close'] / group['Close'].shift(12)
    
    # 'WILLR' – Williams %R (using period 14)
    group['WILLR'] = WILLR(group['High'], group['Low'], group['Close'], period=14)
                            
    # 'TRIX' – Triple Exponential Moving Average Rate Of Change (using period 15)
    group['TRIX'] = TRIX(group['Close'], period=15)
    
    features_list.append(group)

In [ ]:
df = pd.concat(features_list).reset_index().rename(columns={'index': 'Date'})
df = df.sort_values(['Symbol', 'Date']).reset_index(drop=True)

# Outliers

In [ ]:
# First, calculate z-scores if you haven't already
def add_z_score(group):
    mean = group['Price'].mean()
    std = group['Price'].std()
    group['z_score'] = (group['Price'] - mean) / std
    return group

df.reset_index(inplace=True) 

df = df.groupby('Symbol', group_keys=False).apply(add_z_score)

In [ ]:
symbols = df['Symbol'].unique()

for symbol in symbols:
    df_symbol = df[df['Symbol'] == symbol]

    plt.figure(figsize=(14, 6))
    plt.plot(df_symbol['Date'], df_symbol['Price'], color='blue', label='Price')

    # Identify and plot Outliers (z_score > ±3)
    outliers = df_symbol[np.abs(df_symbol['z_score']) > 3]
    plt.scatter(outliers['Date'], outliers['Price'], color='red', label='Outliers')

    plt.title(f'Price and Outliers for Symbol: {symbol}')
    plt.xlabel('Date')
    plt.ylabel('Price')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# # remove outliers
# df = df[df['z_score'].abs() < 3].copy()
# df.drop(columns=['z_score'], inplace=True)
# df.set_index(['Symbol', 'Date'], inplace=True)

# Further Processing

In [ ]:
df.isna().sum()[df.isna().sum() > 0]

In [ ]:
# Fill missing values with forward and backward fill
df = df.ffill().bfill()

In [ ]:
df.columns

In [ ]:
len(df)

In [ ]:
df.to_csv('processed_stock_data.csv', index=False)

# Micro/Marco Indicators

In [ ]:
def clean_macro_csv(file_path):
    df = pd.read_csv(file_path, skiprows=4)
    df.dropna(axis=1, how='all', inplace=True)

    df_long = df.melt(
        id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
        var_name="Year",
        value_name="Value"
    )
    
    df_long["Year"] = pd.to_numeric(df_long["Year"], errors="coerce").astype('Int64')
    df_long["Value"] = pd.to_numeric(df_long["Value"], errors="coerce")
    
    df_long = df_long.reset_index(drop=True)
    
    return df_long

gdp_file = "stck_data/Chỉ số vĩ mô/GDP/GDP.csv"
gdp_growth_file = 'stck_data/Chỉ số vĩ mô/GDP_growth/GDP_growth.csv'
gdp_per_capita_file = 'stck_data/Chỉ số vĩ mô/GDP_per_capita/GDP_per_capita.csv'
inflation_file = 'stck_data/Chỉ số vĩ mô/Inflation/Inflation.csv'
unemployment_file = 'stck_data/Chỉ số vĩ mô/Unemployment/Unemployment.csv'

gdp_df = clean_macro_csv(gdp_file)
gdp_growth_df = clean_macro_csv(gdp_growth_file)
gdp_per_capita_df = clean_macro_csv(gdp_per_capita_file)
inflation_df = clean_macro_csv(inflation_file)
unemployment_df = clean_macro_csv(unemployment_file)

target_country = "Viet Nam"

vn_df = gdp_df[gdp_df["Country Name"].str.strip().str.lower() == target_country.lower()]    
vn_gdp_growth = gdp_growth_df[gdp_growth_df["Country Name"].str.strip().str.lower() == target_country.lower()]
vn_gdp_per_capita = gdp_per_capita_df[gdp_per_capita_df["Country Name"].str.strip().str.lower() == target_country.lower()]
vn_inflation = inflation_df[inflation_df["Country Name"].str.strip().str.lower() == target_country.lower()]
vn_unemployment = unemployment_df[unemployment_df["Country Name"].str.strip().str.lower() == target_country.lower()]

In [ ]:
vn_gdp_growth = vn_gdp_growth.dropna()
vn_gdp_per_capita  = vn_gdp_per_capita .dropna()
vn_df = vn_df.dropna()
vn_inflation = vn_inflation.dropna()
vn_unemployment = vn_unemployment.dropna()

In [ ]:
vn_df = vn_df.rename(columns={"Value": "GDP"})
vn_gdp_growth = vn_gdp_growth.rename(columns={"Value": "GDP_Growth"})
vn_gdp_per_capita = vn_gdp_per_capita.rename(columns={"Value": "GDP_Per_Capita"})
vn_inflation = vn_inflation.rename(columns={"Value": "Inflation"})
vn_unemployment = vn_unemployment.rename(columns={"Value": "Unemployment"})

In [ ]:
vn_merged = vn_df[["Year", "GDP"]].merge( # có từ 1985
    vn_gdp_growth[["Year", "GDP_Growth"]], # có từ 1985
    on="Year",
    how="inner"
).merge(
    vn_gdp_per_capita[["Year", "GDP_Per_Capita"]], # có từ 1985
    on="Year",
    how="inner"
).merge(
    vn_inflation[["Year", "Inflation"]], # có từ 1996
    on="Year",
    how="inner"
).merge(
    vn_unemployment[["Year", "Unemployment"]], # có từ 1991
    on="Year",
    how="inner"
)

vn_merged = vn_merged.sort_values("Year").reset_index(drop=True)

In [ ]:
vn_merged

In [ ]:
vn_merged.to_csv("vietnam_macro.csv", index=False)